In [1]:

!pip install transformers datasets scikit-learn gradio -q
!pip install torch==2.5.0+cu118 torchvision==0.20.0+cu118 torchaudio==2.5.0+cu118 --index-url https://download.pytorch.org/whl/cu118 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.3/838.3 MB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 22.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 49.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 85.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 63.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 111.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.9/663.9 MB 811.0 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 MB 14.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.2/128.2 MB 7.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.1/204.1 MB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os
os.kill(os.getpid(), 9)

In [1]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, f1_score

print("All libraries imported!")
print(f"GPU available: {torch.cuda.is_available()}")

All libraries imported!
GPU available: True


In [2]:
dataset = load_dataset("fancyzhx/ag_news")

print("Dataset loaded!")
print(dataset)
print("\nSample entry:")
print(dataset['train'][0])

README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Dataset loaded!
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

Sample entry:
{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}


In [3]:
# AG News has 4 categories:
# 0 = World, 1 = Sports, 2 = Business, 3 = Science/Technology

label_names = ['World', 'Sports', 'Business', 'Science/Technology']

print("Label mapping:")
for i, name in enumerate(label_names):
    print(f"  {i} = {name}")

print(f"\nTraining samples : {len(dataset['train'])}")
print(f"Testing samples  : {len(dataset['test'])}")

# Show a few examples
print("\nSample headlines:")
for i in range(3):
    sample = dataset['train'][i]
    print(f"  Text  : {sample['text'][:80]}...")
    print(f"  Label : {label_names[sample['label']]}")
    print()

Label mapping:
  0 = World
  1 = Sports
  2 = Business
  3 = Science/Technology

Training samples : 120000
Testing samples  : 7600

Sample headlines:
  Text  : Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall ...
  Label : Business

  Text  : Carlyle Looks Toward Commercial Aerospace (Reuters) Reuters - Private investment...
  Label : Business

  Text  : Oil and Economy Cloud Stocks' Outlook (Reuters) Reuters - Soaring crude prices p...
  Label : Business



In [4]:
# Using 2000 train and 500 test samples to keep it fast

small_train = dataset['train'].shuffle(seed=42).select(range(2000))
small_test  = dataset['test'].shuffle(seed=42).select(range(500))

print(f"Small train size : {len(small_train)}")
print(f"Small test size  : {len(small_test)}")

Small train size : 2000
Small test size  : 500


In [5]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

print("Tokenizer loaded!")

# Test the tokenizer
sample_text = "Apple launches new iPhone with AI features"
tokens = tokenizer(sample_text, return_tensors='pt')
print(f"\nSample text: {sample_text}")
print(f"Token IDs  : {tokens['input_ids']}")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizer loaded!

Sample text: Apple launches new iPhone with AI features
Token IDs  : tensor([[  101,  6207, 18989,  2047, 18059,  2007,  9932,  2838,   102]])


In [6]:
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=128
    )

tokenized_train = small_train.map(tokenize_function, batched=True)
tokenized_test  = small_test.map(tokenize_function, batched=True)

tokenized_train.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
tokenized_test.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

print("Tokenization complete!")

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenization complete!


In [7]:
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=4
)

print("BERT model loaded!")
print(f"Model parameters: {model.num_parameters():,}")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERT model loaded!
Model parameters: 109,485,316


In [8]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, predictions)
    f1  = f1_score(labels, predictions, average='weighted')

    return {
        'accuracy': round(acc, 4),
        'f1_score': round(f1, 4)
    }

print("Metrics function ready!")

Metrics function ready!


In [9]:
training_args = TrainingArguments(
    output_dir='./bert_news_classifier',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    report_to='none'
)

print("Training arguments set!")

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Training arguments set!


In [10]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics
)


trainer.train()
print("Training complete!")

Epoch,Training Loss,Validation Loss,Accuracy,F1 Score
1,0.505129,0.375824,0.876000,0.875900
2,0.250504,0.431039,0.882000,0.881700
3,0.109564,0.400616,0.886000,0.886700


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

Training complete!


In [11]:
results = trainer.evaluate()

print("=" * 40)
print("   EVALUATION RESULTS")
print("=" * 40)
print(f"Accuracy : {results['eval_accuracy'] * 100:.2f}%")
print(f"F1 Score : {results['eval_f1_score'] * 100:.2f}%")
print("=" * 40)

Training Loss,Validation Loss,Epoch,Accuracy,F1 Score
0.109564,0.375530,3,0.876000,0.875900


   EVALUATION RESULTS
Accuracy : 87.60%
F1 Score : 87.59%


In [14]:
# Move model to GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

def predict_topic(text):
    label_names = ['World', 'Sports', 'Business', 'Science/Technology']

    inputs = tokenizer(
        text,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=128
    )

    # Move inputs to same device as model (GPU)
    inputs = {key: val.to(device) for key, val in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    predicted_id = torch.argmax(outputs.logits, dim=-1).item()
    confidence   = torch.softmax(outputs.logits, dim=-1).max().item()

    return label_names[predicted_id], round(confidence * 100, 2)

# Test it!
test_headlines = [
    "NASA discovers water on Mars surface",
    "Manchester United wins Premier League title",
    "Apple stock hits all time high after earnings report",
    "UN holds emergency meeting on Middle East crisis"
]

print("Testing custom headlines:")
print("-" * 50)
for headline in test_headlines:
    topic, confidence = predict_topic(headline)
    print(f"Headline  : {headline}")
    print(f"Predicted : {topic} ({confidence}% confidence)")
    print()

Testing custom headlines:
--------------------------------------------------
Headline  : NASA discovers water on Mars surface
Predicted : Science/Technology (93.22% confidence)

Headline  : Manchester United wins Premier League title
Predicted : Sports (96.02% confidence)

Headline  : Apple stock hits all time high after earnings report
Predicted : Business (91.46% confidence)

Headline  : UN holds emergency meeting on Middle East crisis
Predicted : World (97.57% confidence)



In [15]:
import gradio as gr

def gradio_predict(text):
    topic, confidence = predict_topic(text)
    return f"Topic: {topic}\nConfidence: {confidence}%"

demo = gr.Interface(
    fn=gradio_predict,
    inputs=gr.Textbox(
        lines=3,
        placeholder="Enter a news headline here...",
        label="News Headline"
    ),
    outputs=gr.Textbox(label="Predicted Topic"),
    title="📰 News Topic Classifier (BERT)",
    description="Enter any news headline and BERT will classify it into: World, Sports, Business, or Science/Technology",
    examples=[
        ["Scientists discover new species of dinosaur in Argentina"],
        ["Federal Reserve raises interest rates by 0.5 percent"],
        ["LeBron James announces retirement from NBA"],
        ["China launches new space station module"]
    ]
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cf89ab6e5d05a72496.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [16]:
print("=" * 50)
print("   TASK 1: NEWS TOPIC CLASSIFIER — SUMMARY")
print("=" * 50)
print(f"\nDataset        : AG News (fancyzhx/ag_news)")
print(f"Model          : bert-base-uncased")
print(f"Training Size  : {len(small_train)} samples")
print(f"Testing Size   : {len(small_test)} samples")
print(f"Categories     : World, Sports, Business, Science/Tech")
print(f"\nResults:")
print(f"  Accuracy : {results['eval_accuracy'] * 100:.2f}%")
print(f"  F1 Score : {results['eval_f1_score'] * 100:.2f}%")
print(f"\nDeployment     : Gradio Web Interface ✅")
print("=" * 50)

   TASK 1: NEWS TOPIC CLASSIFIER — SUMMARY

Dataset        : AG News (fancyzhx/ag_news)
Model          : bert-base-uncased
Training Size  : 2000 samples
Testing Size   : 500 samples
Categories     : World, Sports, Business, Science/Tech

Results:
  Accuracy : 87.60%
  F1 Score : 87.59%

Deployment     : Gradio Web Interface ✅
